# 01 — Problem Framing and Six-Class Diagnostics

## Purpose

This notebook investigates the original six-class dataset before the project focuses on five classes. It combines data-quality checks, six-class baselines, error analysis, TF-IDF lexical similarity, optional sentence-embedding similarity, and a blinded label-audit workflow.

**Scope rationale:** the final application focus is on religion-, age-, gender-, and ethnicity-targeted cyberbullying, alongside non-bullying content. Excluding `other_cyberbullying` is a deliberate scope choice—not a claim that all examples in that category are harmless or incorrectly labelled.

**Important interpretation rule:** model errors and similarity scores can motivate investigation, but they do not by themselves prove that a dataset label is invalid.

## Notebook roadmap

1. Load and audit the original data
2. Build the shared, leakage-conscious 70/15/15 split
3. Inspect the six original classes
4. Run six-class TF-IDF + Naive Bayes and Logistic Regression controls
5. Compare model errors on retained classes
6. Measure train-only TF-IDF centroid similarity
7. Optionally run sentence-embedding diagnostics
8. Export a blinded human-review sample
9. Summarize the scope decision and limitations

## Environment

Install the packages in `requirements.txt` from the repository root. The optional sentence-embedding section requires the extra package in `requirements-optional-similarity.txt` and downloads a pretrained model when enabled.

The dataset is not included in this repository. Place `cyberbullying_tweets.csv` in `data/`, or upload it to `/content/` in Google Colab.

In [ ]:
from pathlib import Path
import hashlib
import random
import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

from IPython.display import display

# `contractions` is listed in requirements.txt. Keep a small fallback so the
# notebook remains usable if a fresh environment has not installed it yet.
try:
    from contractions import fix as fix_contractions
except ImportError:
    def fix_contractions(text):
        text = str(text)
        replacements = [
            (r"\bcan't\b", "cannot"), (r"\bwon't\b", "will not"),
            (r"\bshan't\b", "shall not"), (r"\bain't\b", "is not"),
            (r"n['’]t\b", " not"), (r"['’]re\b", " are"),
            (r"['’]ve\b", " have"), (r"['’]ll\b", " will"),
            (r"['’]d\b", " would"), (r"['’]m\b", " am"),
        ]
        for pattern, replacement in replacements:
            text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
        return text

SEED = 2042
ALL_LABELS = [
    "religion", "age", "ethnicity", "gender",
    "not_cyberbullying", "other_cyberbullying"
]
RETAINED_LABELS = [
    "religion", "age", "ethnicity", "gender", "not_cyberbullying"
]
OTHER_LABEL = "other_cyberbullying"

random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 140)

print("Random seed:", SEED)

## 1. Load, inspect, and make the shared split

All three notebooks use the same canonical source-row filtering and deterministic split code. The `split fingerprint` printed below should match in the TF-IDF and LSTM notebooks. Exact/normalized duplicate handling is performed before splitting to reduce duplicate-text leakage across partitions.

In [ ]:
# Locate the dataset whether this notebook is run from the repository root,
# the notebooks/ folder, or Google Colab after uploading the CSV.
CANDIDATE_PATHS = [
    Path("data/cyberbullying_tweets.csv"),
    Path("../data/cyberbullying_tweets.csv"),
    Path("cyberbullying_tweets.csv"),
    Path("/content/cyberbullying_tweets.csv"),
]
DATA_PATH = next((p for p in CANDIDATE_PATHS if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find cyberbullying_tweets.csv. Place it at data/cyberbullying_tweets.csv "
        "or upload it to /content/cyberbullying_tweets.csv in Colab."
    )

raw = pd.read_csv(DATA_PATH).rename(columns={
    "tweet_text": "text",
    "cyberbullying_type": "label",
})
assert {"text", "label"}.issubset(raw.columns), (
    "Expected columns text/label or tweet_text/cyberbullying_type."
)

raw = raw.dropna(subset=["text", "label"]).copy()
raw["text"] = raw["text"].astype(str).str.strip()
raw["label"] = raw["label"].astype(str).str.strip().str.lower()
raw["label"] = raw["label"].replace({
    "not bullying": "not_cyberbullying",
    "not cyberbullying": "not_cyberbullying",
    "other bullying": "other_cyberbullying",
})
raw = raw[raw["text"].ne("") & raw["label"].isin(ALL_LABELS)].copy()

print(f"Dataset path: {DATA_PATH}")
print(f"Rows after basic missing/label checks: {len(raw):,}")
print(f"Raw exact duplicate rows: {raw.duplicated(subset=['text', 'label']).sum():,}")

# Conservative, shared deduplication key. This is deliberately separate from
# either model's actual text-cleaning function. It reduces the chance that
# punctuation/URL/mention variants of the same tweet cross data partitions.
def make_split_key(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " URLTOKEN ", text)
    text = re.sub(r"@\w+", " MENTIONTOKEN ", text)
    text = re.sub(r"#([\w-]+)", r"\1", text)
    text = re.sub(r"[^\w\s]", " ", text)
    return " ".join(text.split()).strip()

raw["split_key"] = raw["text"].map(make_split_key)
raw = raw[raw["split_key"].ne("")].copy()

# If the same normalized key carries more than one original label, preserve
# those rows for inspection but exclude that key from model evaluation.
label_count_per_key = raw.groupby("split_key")["label"].nunique()
conflicting_keys = set(label_count_per_key[label_count_per_key > 1].index)
label_conflicts = raw[raw["split_key"].isin(conflicting_keys)].copy()

model_source = raw[~raw["split_key"].isin(conflicting_keys)].copy()
rows_before_dedup = len(model_source)
model_df = model_source.drop_duplicates(subset="split_key", keep="first").copy()
model_df = model_df.reset_index(drop=True)
model_df["row_id"] = np.arange(len(model_df), dtype=np.int64)

print(f"Normalized text keys with conflicting labels: {len(conflicting_keys):,}")
print(f"Rows held aside due to conflicting labels: {len(label_conflicts):,}")
print(f"Repeated normalized keys removed: {rows_before_dedup - len(model_df):,}")
print(f"Rows available for modelling: {len(model_df):,}")

if not label_conflicts.empty:
    print("Examples of conflicting normalized keys (inspect rather than assume annotation error):")
    display(label_conflicts[["text", "label", "split_key"]].head(10))

observed_labels = set(model_df["label"].unique())
assert observed_labels == set(ALL_LABELS), (
    f"Expected all six labels {ALL_LABELS}, found {sorted(observed_labels)}"
)

# One shared, deterministic 70/15/15 split from the ORIGINAL six-class data.
# Filter out 'other_cyberbullying' only after these partitions are created.
train_df, temp_df = train_test_split(
    model_df,
    test_size=0.30,
    random_state=SEED,
    stratify=model_df["label"],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"],
)

split_parts = {"train": train_df, "validation": val_df, "test": test_df}
print(f"Train: {len(train_df):,} ({len(train_df)/len(model_df):.1%})")
print(f"Validation: {len(val_df):,} ({len(val_df)/len(model_df):.1%})")
print(f"Test: {len(test_df):,} ({len(test_df)/len(model_df):.1%})")

split_counts = pd.DataFrame({
    name: part["label"].value_counts()
    for name, part in split_parts.items()
}).fillna(0).astype(int).reindex(ALL_LABELS)
display(split_counts)

# This fingerprint should match across all three notebooks when using the same CSV.
fp_payload = "|".join(
    f"{name}:{','.join(map(str, sorted(part['row_id'].tolist())))}"
    for name, part in split_parts.items()
)
split_fingerprint = hashlib.sha256(fp_payload.encode("utf-8")).hexdigest()[:12]
print("Shared split fingerprint:", split_fingerprint)
print("Record this fingerprint when comparing results from the three notebooks.")

## 2. Original six-class distribution

The original categories are kept intact for this diagnostic. We first examine the data and model errors before deciding the scope of the final model.

In [ ]:
class_counts = (
    model_df["label"].value_counts()
    .reindex(ALL_LABELS)
    .rename_axis("label")
    .reset_index(name="count")
)
display(class_counts)

plt.figure(figsize=(10, 4))
plt.bar(class_counts["label"], class_counts["count"])
plt.xticks(rotation=30, ha="right")
plt.title("Original six-class distribution after shared duplicate handling")
plt.xlabel("")
plt.ylabel("Tweets")
plt.tight_layout()
plt.show()

## 3. Classical preprocessing

This function is also used in Notebook 02, ensuring the same TF-IDF preprocessing across the six-class control and five-class final model. Negation words are preserved because removing `not` can reverse the meaning of a sentence.

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import warnings

# Use the NLTK lists when available; fall back to scikit-learn's built-in list
# if the corpus download is unavailable in a restricted environment.
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
try:
    STOP_WORDS = set(stopwords.words("english"))
except LookupError:
    warnings.warn("NLTK stopwords corpus unavailable; using scikit-learn stopwords.")
    STOP_WORDS = set(ENGLISH_STOP_WORDS)

# Preserve negation: removing 'not' can reverse the meaning of a tweet.
STOP_WORDS -= {"no", "nor", "not"}
LEMMATIZER = WordNetLemmatizer()
try:
    LEMMATIZER.lemmatize("validation")
    WORDNET_AVAILABLE = True
except LookupError:
    WORDNET_AVAILABLE = False
    warnings.warn("WordNet corpus unavailable; lemmatization will be skipped.")

def clean_tweet_classical(text):
    """Lightly normalize a tweet for a TF-IDF model; preserve negation words."""
    text = "" if pd.isna(text) else str(text)
    text = fix_contractions(text)
    text = re.sub(r"\r|\n", " ", text.lower())
    text = re.sub(r"https?://\S+|www\.\S+", " URLTOKEN ", text)
    text = re.sub(r"@\w+", " MENTIONTOKEN ", text)
    text = re.sub(r"#([\w-]+)", r"\1", text)
    text = re.sub(r"[^\x00-\x7F]", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", " ", text)

    tokens = [w for w in text.split() if w not in STOP_WORDS and len(w) >= 2]
    tokens = [LEMMATIZER.lemmatize(w) for w in tokens] if WORDNET_AVAILABLE else tokens
    text = " ".join(tokens)
    text = re.sub(r"\b(\w+)((\w)\3{2,})(\w*)\b", r"\1\3\4", text)
    text = " ".join(text.split()).strip()
    return text if text else "emptytext"

print("Classical preprocessing ready. Negations 'no', 'nor', 'not' are preserved.")

## 4. Six-class TF-IDF baselines

TF-IDF is fitted only on the training texts. Oversampling is applied only to the training feature matrix. The validation set selects Logistic Regression's `C`; the test set is used once for final reporting.

These settings intentionally provide a reproducible baseline rather than an exhaustive hyperparameter search.

In [ ]:
from imblearn.over_sampling import RandomOverSampler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, precision_recall_fscore_support
)

train_text_6 = train_df["text"].map(clean_tweet_classical).tolist()
val_text_6 = val_df["text"].map(clean_tweet_classical).tolist()
test_text_6 = test_df["text"].map(clean_tweet_classical).tolist()
y_train_6 = train_df["label"].to_numpy()
y_val_6 = val_df["label"].to_numpy()
y_test_6 = test_df["label"].to_numpy()

vectorizer_6 = TfidfVectorizer(
    ngram_range=(1, 1),
    min_df=1,
    max_features=50000,
    sublinear_tf=False,
)
X_train_6 = vectorizer_6.fit_transform(train_text_6)
X_val_6 = vectorizer_6.transform(val_text_6)
X_test_6 = vectorizer_6.transform(test_text_6)

ros_6 = RandomOverSampler(random_state=SEED)
X_train_6_bal, y_train_6_bal = ros_6.fit_resample(X_train_6, y_train_6)

print("Train TF-IDF shape:", X_train_6.shape)
print("Balanced training shape:", X_train_6_bal.shape)
print("Validation shape:", X_val_6.shape)
print("Test shape:", X_test_6.shape)

In [ ]:
# Baseline 1: Multinomial Naive Bayes.
nb_6 = MultinomialNB()
nb_6.fit(X_train_6_bal, y_train_6_bal)
nb_pred_6 = nb_6.predict(X_test_6)

print("Six-class Naive Bayes")
print(f"Accuracy: {accuracy_score(y_test_6, nb_pred_6):.4f}")
print(f"Macro-F1: {f1_score(y_test_6, nb_pred_6, average='macro'):.4f}")
print(classification_report(
    y_test_6, nb_pred_6,
    labels=ALL_LABELS,
    target_names=ALL_LABELS,
    digits=3,
    zero_division=0,
))

In [ ]:
# Baseline 2: Logistic Regression. Select C using validation macro-F1.
C_CANDIDATES = [0.5, 1.0, 2.0]
validation_rows = []
models_6_by_c = {}

for c in C_CANDIDATES:
    candidate = LogisticRegression(
        C=c,
        max_iter=2000,
        random_state=SEED,
    )
    candidate.fit(X_train_6_bal, y_train_6_bal)
    val_pred = candidate.predict(X_val_6)
    val_macro_f1 = f1_score(y_val_6, val_pred, average="macro")
    validation_rows.append({"C": c, "validation_macro_f1": val_macro_f1})
    models_6_by_c[c] = candidate

validation_results_6 = pd.DataFrame(validation_rows).sort_values(
    "validation_macro_f1", ascending=False
)
display(validation_results_6)
BEST_C_6 = float(validation_results_6.iloc[0]["C"])
lr_6 = models_6_by_c[BEST_C_6]
lr_pred_6 = lr_6.predict(X_test_6)

six_class_summary = pd.DataFrame([{
    "model": "TF-IDF + Logistic Regression (6 classes)",
    "selected_C": BEST_C_6,
    "accuracy": accuracy_score(y_test_6, lr_pred_6),
    "macro_f1": f1_score(y_test_6, lr_pred_6, average="macro"),
}])
display(six_class_summary.round(4))
print(classification_report(
    y_test_6, lr_pred_6,
    labels=ALL_LABELS,
    target_names=ALL_LABELS,
    digits=3,
    zero_division=0,
))

In [ ]:
cm_6 = confusion_matrix(y_test_6, lr_pred_6, labels=ALL_LABELS)
cm_6_normalized = confusion_matrix(
    y_test_6, lr_pred_6, labels=ALL_LABELS, normalize="true"
)

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
sns.heatmap(cm_6, annot=True, fmt="d", cmap="Blues",
            xticklabels=ALL_LABELS, yticklabels=ALL_LABELS, ax=axes[0])
axes[0].set_title("Six-class LR — counts")
axes[0].set_xlabel("Predicted label")
axes[0].set_ylabel("True label")
sns.heatmap(cm_6_normalized, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
            xticklabels=ALL_LABELS, yticklabels=ALL_LABELS, ax=axes[1])
axes[1].set_title("Six-class LR — row-normalized")
axes[1].set_xlabel("Predicted label")
axes[1].set_ylabel("True label")
for ax in axes:
    ax.tick_params(axis="x", rotation=35)
    ax.tick_params(axis="y", rotation=0)
plt.tight_layout()
plt.show()

## 5. A controlled look at the five retained test classes

The next table evaluates the six-class model only on test tweets whose **true labels are among the five retained classes**. A prediction of `other_cyberbullying` still counts as an error for those rows. This is a useful control for Notebook 02 because both formulations can be evaluated on the same retained test examples.

It does not yet compare against the five-class model; that model is trained and evaluated in Notebook 02.

In [ ]:
retained_test_mask = np.isin(y_test_6, RETAINED_LABELS)
y_test_retained = y_test_6[retained_test_mask]
y_pred_6_on_retained = lr_pred_6[retained_test_mask]

p, r, f, support = precision_recall_fscore_support(
    y_test_retained,
    y_pred_6_on_retained,
    labels=RETAINED_LABELS,
    zero_division=0,
)
retained_control_6 = pd.DataFrame({
    "class": RETAINED_LABELS,
    "precision": p,
    "recall": r,
    "f1": f,
    "support": support,
})
display(retained_control_6.round(3))
print("Accuracy on the retained-label subset:",
      round(accuracy_score(y_test_retained, y_pred_6_on_retained), 4))
print("Macro-F1 on the retained-label subset:",
      round(f1_score(y_test_retained, y_pred_6_on_retained,
                     labels=RETAINED_LABELS, average="macro"), 4))
print("Number of retained-class test rows:", len(y_test_retained))

## 6. TF-IDF lexical centroid similarity

This is a **train-only diagnostic**. It measures similarity between average word-weighted vectors; it does not prove semantic equivalence or label invalidity. To keep it lightweight, we sample up to 3,000 training tweets per class.

In [ ]:
diag_base = pd.DataFrame({
    "text_clean": train_text_6,
    "label": y_train_6,
})
diag_parts = []
for label, group in diag_base.groupby("label"):
    diag_parts.append(group.sample(n=min(len(group), 3000), random_state=SEED))
diag_df = pd.concat(diag_parts, ignore_index=True)

diag_vectorizer = TfidfVectorizer(
    min_df=2,
    max_features=15000,
    ngram_range=(1, 1),
)
X_diag = diag_vectorizer.fit_transform(diag_df["text_clean"])
labels_diag = [label for label in ALL_LABELS if label in set(diag_df["label"])]
centroids = []
for label in labels_diag:
    rows = X_diag[diag_df["label"].to_numpy() == label]
    centroids.append(np.asarray(rows.mean(axis=0)).ravel())

lexical_sim = cosine_similarity(np.vstack(centroids))
lexical_sim_df = pd.DataFrame(lexical_sim, index=labels_diag, columns=labels_diag)
display(lexical_sim_df.round(3))

plt.figure(figsize=(9, 7))
sns.heatmap(lexical_sim_df, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1)
plt.title("Train-only TF-IDF class-centroid similarity")
plt.tight_layout()
plt.show()

other_row = lexical_sim_df.loc[OTHER_LABEL].drop(OTHER_LABEL).sort_values(ascending=False)
print("Classes lexically most similar to other_cyberbullying:")
display(other_row.rename("cosine_similarity").to_frame())

## 7. Optional semantic-embedding diagnostic

Earlier exploratory variants used different sample sizes and preprocessing. To avoid mixing results from different pipelines, this version fits the diagnostic on a balanced sample from the **training partition only**, using raw tweet text. It is optional because it downloads a pretrained model and adds runtime.

Set `RUN_SENTENCE_EMBEDDING_DIAGNOSTIC = True` when you want to reproduce it. High centroid similarity is still supporting evidence—not proof that two labels mean the same thing.

In [ ]:
RUN_SENTENCE_EMBEDDING_DIAGNOSTIC = False
MAX_SENTENCES_PER_CLASS = 1000

if RUN_SENTENCE_EMBEDDING_DIAGNOSTIC:
    try:
        from sentence_transformers import SentenceTransformer
        from sklearn.metrics.pairwise import cosine_similarity
    except ImportError as exc:
        raise ImportError(
            "Install requirements-optional-similarity.txt to run this section."
        ) from exc

    sem_parts = []
    for label, group in train_df.groupby("label"):
        sem_parts.append(group.sample(
            n=min(len(group), MAX_SENTENCES_PER_CLASS), random_state=SEED
        ))
    sem_df = pd.concat(sem_parts, ignore_index=True)

    encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embeddings = encoder.encode(
        sem_df["text"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    y_sem = sem_df["label"].to_numpy()
    labels_sem = [label for label in ALL_LABELS if label in set(y_sem)]
    semantic_centroids = []
    for label in labels_sem:
        centroid = embeddings[y_sem == label].mean(axis=0)
        norm = np.linalg.norm(centroid)
        semantic_centroids.append(centroid / norm if norm else centroid)

    semantic_sim = cosine_similarity(np.vstack(semantic_centroids))
    semantic_sim_df = pd.DataFrame(
        semantic_sim, index=labels_sem, columns=labels_sem
    )
    display(semantic_sim_df.round(3))

    plt.figure(figsize=(9, 7))
    sns.heatmap(semantic_sim_df, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1)
    plt.title("Train-only sentence-embedding class-centroid similarity")
    plt.tight_layout()
    plt.show()

    # Nearest-neighbor examples: keep this small to limit memory usage.
    QUERY_PER_CLASS = 200
    query_df = sem_df[sem_df["label"] == OTHER_LABEL].sample(
        n=min(QUERY_PER_CLASS, int((sem_df["label"] == OTHER_LABEL).sum())),
        random_state=SEED,
    )
    candidate_df = sem_df[sem_df["label"] != OTHER_LABEL]
    query_pos = query_df.index.to_numpy()
    candidate_pos = candidate_df.index.to_numpy()
    query_embeddings = embeddings[query_pos]
    candidate_embeddings = embeddings[candidate_pos]
    scores = query_embeddings @ candidate_embeddings.T
    best_positions = scores.argmax(axis=1)
    nearest_examples = pd.DataFrame({
        "other_tweet": query_df["text"].to_numpy(),
        "nearest_tweet": candidate_df.iloc[best_positions]["text"].to_numpy(),
        "nearest_label": candidate_df.iloc[best_positions]["label"].to_numpy(),
        "cosine_similarity": scores[np.arange(len(best_positions)), best_positions],
    })
    display(nearest_examples.sort_values("cosine_similarity", ascending=False).head(25))
else:
    print("Skipped. Set RUN_SENTENCE_EMBEDDING_DIAGNOSTIC = True to reproduce the embedding diagnostic.")

## 8. Blinded human label-audit workflow

This is the most direct next step for investigating the original label definitions. A similarity score or classifier error does not establish the correct label. The goal is to have reviewers independently judge whether examples labelled `other_cyberbullying` fit one of the retained categories or are genuinely distinct/ambiguous.

**Reviewer procedure:** hide original labels; provide a short written label guide; ask two reviewers to label independently; record brief notes. Use exactly one label per tweet: `religion`, `age`, `ethnicity`, `gender`, `not_cyberbullying`, `other_cyberbullying`, or `ambiguous_insufficient_context`. Keep the private key out of GitHub.

In [ ]:
# Export a pilot blind-review sheet (maximum 400 tweets).
N_OTHER_FOR_AUDIT = 150
N_PER_OTHER_CLASS = 50

audit_parts = []
for label, group in model_df.groupby("label"):
    sample_n = N_OTHER_FOR_AUDIT if label == OTHER_LABEL else N_PER_OTHER_CLASS
    audit_parts.append(group.sample(n=min(len(group), sample_n), random_state=SEED))
audit_source_sample = pd.concat(audit_parts).sample(frac=1, random_state=SEED).reset_index(drop=True)
audit_source_sample.insert(0, "audit_id", [f"A{i:04d}" for i in range(1, len(audit_source_sample) + 1)])

audit_key_private = audit_source_sample[["audit_id", "label"]].rename(
    columns={"label": "original_label"}
)
blind_review = audit_source_sample[["audit_id", "text"]].copy()
blind_review["reviewer_1_label"] = ""
blind_review["reviewer_2_label"] = ""
blind_review["notes"] = ""

blind_review.to_csv("cyberbullying_blind_review.csv", index=False)
audit_key_private.to_csv("cyberbullying_audit_key_PRIVATE.csv", index=False)
print("Created cyberbullying_blind_review.csv and cyberbullying_audit_key_PRIVATE.csv")
print("Do not commit the private key, the completed sheet, or raw tweet examples to a public repository.")
display(blind_review.head())

### Analyze a completed review (run only after reviewers fill the sheet)

Expected reviewer labels are the six original categories plus `ambiguous_insufficient_context`. This cell runs only if both local review files exist.

In [ ]:
from pathlib import Path
from sklearn.metrics import cohen_kappa_score

completed_path = Path("cyberbullying_blind_review_completed.csv")
key_path = Path("cyberbullying_audit_key_PRIVATE.csv")

if completed_path.exists() and key_path.exists():
    completed = pd.read_csv(completed_path)
    key = pd.read_csv(key_path)
    reviewed = completed.merge(key, on="audit_id", validate="one_to_one")
    reviewer_cols = ["reviewer_1_label", "reviewer_2_label"]
    for col in reviewer_cols:
        reviewed[col] = reviewed[col].astype("string").str.strip().str.lower()
        reviewed[col] = reviewed[col].replace({
            "not bullying": "not_cyberbullying",
            "not cyberbullying": "not_cyberbullying",
            "other bullying": "other_cyberbullying",
        })
    valid = reviewed.dropna(subset=reviewer_cols)
    valid = valid[(valid[reviewer_cols[0]] != "") & (valid[reviewer_cols[1]] != "")]

    print("Reviewed rows:", len(valid))
    print("Reviewer agreement (Cohen's kappa):",
          round(cohen_kappa_score(valid[reviewer_cols[0]], valid[reviewer_cols[1]]), 3))

    for col in reviewer_cols:
        print(f"\n{col}: original other_cyberbullying examples")
        subset = valid[valid["original_label"] == OTHER_LABEL]
        display(subset[col].value_counts(normalize=True).mul(100).round(1).rename("percent").to_frame())
else:
    print("No completed review sheet found. Complete the exported CSV first; keep the private audit key local.")

## 9. Interpretation and scope decision

### What this notebook can support

- Whether the tested six-class models struggle with specific categories on held-out data.
- Which classes have overlapping lexical or average sentence representations in the training data.
- Whether normalized duplicate examples appear with conflicting labels.
- Whether independent human reviewers judge sampled `other_cyberbullying` examples as fitting retained categories, genuinely distinct, or ambiguous (if the audit is completed).

### What it cannot prove by itself

- A higher five-class score does not prove the removed class is invalid; excluding a class makes the task narrower.
- Cosine similarity does not prove that labels are semantically equivalent.
- The dataset labels do not determine whether a tweet is a crime or what moderation action is legally required.

### Portfolio conclusion

The project focuses on four identity-targeted categories—religion, age, gender, and ethnicity—alongside non-bullying content. The six-class experiments and similarity diagnostics motivate investigating that focused scope. The choice to exclude `other_cyberbullying` is described as a scope decision unless the human audit supplies stronger label-quality evidence.